# Avance 4 — Modelos alternativos

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Compara familias de modelos **individuales** sobre las dos fases del proyecto, ajusta las dos
mejores y elige una. Parte del nivel de referencia del [Avance 3](03_Baseline.html) y usa la
misma partición, las mismas métricas y el mismo arnés, para que las cifras sean comparables.

## Qué produce

1. Una **comparativa de seis familias sin ensamble**, ordenada por la métrica principal, con
   otras métricas y los tiempos de entrenamiento.
2. El **ajuste de hiperparámetros** de las dos mejores, con la búsqueda **dentro** del pliegue.
3. La **estabilidad** de lo que esa búsqueda eligió entre pliegues.
4. El **modelo individual final** y el argumento de por qué ese.

## Por qué no aparecen aquí el bosque ni el boosting

La rúbrica pide seis modelos «individuales, no ensambles», y la del Avance 5 pide ensambles
homogéneos y heterogéneos. Un bosque aleatorio y un boosting son ensambles homogéneos por
construcción, así que pertenecen al Avance 5. Se incluyen en la tabla como referencia del
Avance 3, marcados y fuera del conteo de seis.

## 1. Preparación

In [ ]:
import sys, warnings, platform
warnings.filterwarnings("ignore")
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Se inserta la raíz y no `src/`: `src.models` importa de `src.data_prep`, que a su vez
# importa de `src.features`. Es la convención del 02b, el 03 y las pruebas.
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from src.evaluation.cv import evaluar, intervalo_por_grupos, resumen, tabla_comparativa
from src.evaluation.graficos import (concordantes, curva_precision_exhaustividad,
                                     curva_roc, importancias_dos_metodos, matriz_confusion)
from src.models import BusquedaAnidada, cargar, canalizaciones, catalogo, seleccion

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 120

In [ ]:
# Versiones con las que se ejecutó este notebook. `requirements.txt` fija solo cotas
# inferiores y entre versiones cambian las cifras en el tercer decimal, así que sin esta
# línea una salida no es rastreable a su entorno. Tiene que decir Python 3.12: es la
# versión del lock y la que ejecutó el resto del proyecto.
import scipy, rdkit, matplotlib, seaborn, joblib

print(f"sistema      {platform.platform()}")
print(f"python       {platform.python_version()}")
for m in (np, pd, scipy, sklearn, rdkit, matplotlib, seaborn, joblib):
    print(f"{m.__name__:<12} {m.__version__}")

## 2. Las seis familias

El catálogo de `src/models/alternativos.py` ofrece ocho familias sin ensamble con sesgos
inductivos distintos. Elegir seis es decisión del equipo y hay que poder explicar por qué
esas: la rúbrica evalúa que los algoritmos sean variados, y la reunión semanal pregunta por
el criterio.

In [ ]:
# DECISIÓN DEL EQUIPO (H1). Seis claves de `FAMILIAS`:
# logistica, lda, arbol, knn, bayes, svm_rbf, svm_sgd, mlp
SEIS = ()

# DECISIÓN DEL EQUIPO (H2). La métrica que ordena la tabla, coherente con la C.1.
PRINCIPAL = "mcc"

if len(SEIS) != 6:
    raise ValueError(
        "Elijan seis familias del catálogo antes de correr el resto (H1 del plan). "
        "La lista vacía es deliberada: un valor por omisión tomaría la decisión en "
        "silencio y nadie podría explicarla en la reunión.")
modelos = seleccion(SEIS)

## 3. Comparativa

Las seis familias con el mismo arnés del Avance 3: partición agrupada por andamio y esqueleto,
tres semillas, cinco pliegues. Los tiempos son de reloj de pared, así que para que la columna
signifique lo mismo en todas las filas hay que correr con el mismo paralelismo.

In [ ]:
tarea1 = cargar("permeabilidad")
res1, fdp1 = evaluar(modelos, tarea1.X, tarea1.y, tarea1.grupos)
tabla1 = tabla_comparativa(res1, principal=PRINCIPAL)
tabla1

> **Pendiente del equipo.** Leer la tabla de la fase 1. Qué familia encabeza, si la distancia a la segunda es mayor que la dispersión, y qué cuesta en tiempo.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

In [ ]:
# Los dos ensambles del Avance 3, como referencia y fuera del conteo de seis.
ref1, _ = evaluar(canalizaciones(), tarea1.X, tarea1.y, tarea1.grupos)
tabla_comparativa(ref1, principal=PRINCIPAL)

> **Pendiente del equipo.** Comparar las seis familias individuales contra los dos ensambles del Avance 3, sin contarlos entre los seis.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

### 3.1 Fase 2 · Actividad en el sistema nervioso

La fase 2 tiene 54,369 pares y la unidad es el par compuesto-diana. `svm_rbf` crece entre
cuadrático y cúbico: un solo ajuste sobre el pliegue completo ronda los cinco minutos, así
que si está entre las seis hay que decidir si se corre aquí o se declara su costo.

In [ ]:
tarea2 = cargar("actividad", umbral_um=10.0, representacion="descriptores+diana")
res2, fdp2 = evaluar(modelos, tarea2.X, tarea2.y, tarea2.grupos, estratos=tarea2.estratos)
tabla_comparativa(res2, principal=PRINCIPAL)

> **Pendiente del equipo.** Leer la fase 2, y el desglose por procedencia: un agregado no describe a los productos naturales (decisión 2.3).
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 4. Ajuste fino de las dos mejores

La búsqueda va **dentro** del pliegue y su validación interna también agrupa. Hacerlo con una
partición aleatoria dejaría la evaluación externa intacta y elegiría la rejilla sobre una
medición inflada entre 7 y 13 puntos, que es una fuga que no aparece en ninguna métrica.

In [ ]:
# DECISIÓN DEL EQUIPO (H3). Rejilla por familia y presupuesto de búsqueda.
REJILLAS = {}

if not REJILLAS:
    raise ValueError(
        "Definan la rejilla de las dos mejores antes de correr (H3 del plan). "
        "Qué rangos se exploran y cuántas combinaciones es parte de lo que la "
        "rúbrica califica como profundidad del espacio de búsqueda.")

afinados = {f"{nombre} afinado": BusquedaAnidada(catalogo()[nombre], rejilla, n_jobs=-1)
            for nombre, rejilla in REJILLAS.items()}
res_af, fdp_af = evaluar(afinados, tarea1.X, tarea1.y, tarea1.grupos)
tabla_comparativa(res_af, principal=PRINCIPAL)

> **Pendiente del equipo.** Decir si el ajuste mejoró, cuánto, y si la mejora es mayor que la dispersión entre pliegues.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

### 4.1 Qué eligió la búsqueda en cada pliegue

Una rejilla que cambia de pliegue a pliegue no es un óptimo: es una elección que depende de
la partición. Es la diferencia entre un ajuste fino y una corrida afortunada.

In [ ]:
res_af[["modelo", "semilla", "pliegue", PRINCIPAL, "elegidos"]].to_string(index=False)

> **Pendiente del equipo.** Decir si lo elegido fue estable entre pliegues o cambió.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 5. Gráficos del mejor individual

Sobre predicciones **fuera de pliegue**, no de un modelo reajustado sobre todo el conjunto:
un gráfico hecho sobre datos que el modelo vio describe su memoria, no su desempeño.

In [ ]:
# El mejor individual según la tabla; cambiar la clave si la elección fue otra.
MEJOR = tabla1.index[0]
p = fdp1[MEJOR][0]

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
curva_roc(tarea1.y, p, ax=axes[0, 0], etiqueta=MEJOR)
curva_precision_exhaustividad(tarea1.y, p, ax=axes[0, 1], etiqueta="BBB−")
matriz_confusion(tarea1.y, p, ax=axes[1, 0], nombres=("BBB−", "BBB+"))
importancias_dos_metodos(pd.read_csv(RAIZ / "notebooks/datos/rubrica3_importancias.csv"),
                         ax=axes[1, 1])
fig.tight_layout()

> **Pendiente del equipo.** Interpretar los cuatro gráficos, uno por uno. La rúbrica del Avance 5 pide interpretación detallada, no la figura sola.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 6. El modelo individual final

La rúbrica da el máximo cuando el argumento no es solo la métrica: compensaciones entre
métricas, interpretabilidad, costo de entrenamiento, desempeño sobre los productos naturales.
Si los intervalos se solapan y ninguna familia se distingue, decirlo es el resultado.

> **Pendiente del equipo.** Escribir el argumento del modelo final.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

## 7. Qué queda establecido y qué no

> **Pendiente del equipo.** Cerrar el avance: qué se puede afirmar con la evidencia de esta libreta y qué no.
>
> No redactar antes de ver los números: una frase escrita antes del resultado se acomoda al resultado.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | | | |
| | | | |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | |
| Artemio Santiago Padilla Robles | |
| José Antonio Vázquez Martínez | |

### Decisiones principales

**Decisión 1:**

- Justificación:
- Alternativa descartada y motivo:

**Decisión 2:**

- Justificación:
- Alternativa descartada y motivo:

**Decisión 3:**

- Justificación:
- Alternativa descartada y motivo:

### Responsabilidad

Las salidas de las herramientas se usaron como insumo de trabajo y fueron revisadas,
verificadas, editadas y adaptadas por el equipo. El equipo asume la responsabilidad
por la exactitud, la integridad y el contenido final del avance.